In [ ]:
import gc

import cupy as cp
import torch

MIB = 1024 * 1024

In [ ]:
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
from rmm.allocators.torch import rmm_torch_allocator

pool = rmm.mr.CudaAsyncMemoryResource(
    initial_pool_size=0, release_threshold=1024 * MIB
)
rmm.mr.set_current_device_resource(pool)
torch.cuda.memory.change_current_allocator(rmm_torch_allocator)
cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:
torch.cuda.init()
cp.cuda.runtime.memGetInfo()
torch.cuda.synchronize()
baseline_free = cp.cuda.runtime.memGetInfo()[0]


def held_mib():
    return (baseline_free - cp.cuda.runtime.memGetInfo()[0]) / MIB

In [ ]:
print("allocator:", "rmm")
print("baseline:", held_mib())

tensor = torch.empty(2048 * MIB, dtype=torch.uint8, device="cuda")
print("torch allocated:", held_mib())

del tensor
gc.collect()
print("torch deleted:", held_mib())

torch.cuda.synchronize()
print("synchronized:", held_mib())

torch.cuda.empty_cache()
print("torch empty_cache:", held_mib())